Imports

In [1]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 
from midiutil import MIDIFile
from audiolazy_functions import str2midi, midi2str  # using premade functions from a sonification tutorial to convert notes to MIDI 
from IPython.display import Audio
from music21 import chord, instrument, note, stream

In [2]:
# function taken from sonification tutorial 
def map_value(value, min_value, max_value, min_result, max_result):
    '''maps value (or array of values) from one range to another'''
 
    result = min_result + (value - min_value)/(max_value - min_value)*(max_result - min_result)
    return result

In [ ]:
# data processing
df = pd.read_csv("temp_data.csv", skiprows=7, names=['years', 1,2,3,4,5,6,7,8,9,10,11,12])
processed_df = (df.round())
long_proc_df = processed_df.melt(id_vars='years', var_name="month", value_name='temperature')
long_proc_df['date'] = pd.to_datetime(long_proc_df[['years', 'month']].assign(day=1))
long_proc_df = long_proc_df.sort_values('date')
long_proc_df
temps = np.array(long_proc_df['temperature'])

# yearly average 
yearly_avg = df.drop(columns='years').mean(axis=1).round()
yearly_avg_df = pd.DataFrame({'year': df['years'], 'average_temperature': yearly_avg})
print(yearly_avg_df.to_string(index=False))
print(f"Rounded yearly-average range: {int(yearly_avg.min())} to {int(yearly_avg.max())} °C")


max_temp = max(temps)
min_temp = min(temps)
range_temp = max_temp - min_temp

# scales and midi conversion 
pentatonic = ['C1', 'D1', 'E1', 'G1', 'A1', 'C2', 'D2', 'E2', 'G2', 'A2','C3', 'D3', 'E3', 'G3', 'A3', 'C4', 'D4', 'E4', 'G4', 'A4', 'C5', 'D5', 'E5', 'G5', 'A5', 'C6', 'D6', 'E6', 'G6', 'A6']
lydian = ['C1', 'D1', 'E1', 'F#1', 'G1', 'A1', 'B1', 'C2', 'D2', 'E2', 'F#2', 'G2', 'A2', 'B2','C3', 'D3', 'E3', 'F#3', 'G3', 'A3', 'B3', 'C4', 'D4', 'E4', 'F#4', 'G4', 'A4', 'B4', 'C5', 'D5']
dorian = ['C1', 'D1', 'Eb1', 'F1', 'G1', 'A1', 'Bb1', 'C2', 'D2', 'Eb2', 'F2', 'G2', 'A2', 'Bb2','C3', 'D3', 'Eb3', 'F3', 'G3', 'A3', 'Bb3', 'C4', 'D4', 'Eb4', 'F4', 'G4', 'A4', 'Bb4', 'C5', 'D5']

note_names = lydian
midi_notes = (str2midi(n) for n in note_names)
midi_notes = list(midi_notes)
n_notes = len(note_names)

t_notes = [note.Note(n) for n in note_names]
stream_ = stream.Stream(t_notes)
stream_.show('midi', autoplay=True)


max_midi = max(midi_notes)
min_midi = min(midi_notes)
range_midi = max_midi - min_midi

### Rain, water 

In [ ]:
import base64, uuid
from pathlib import Path
from IPython.display import HTML, display

SOUNDS_DIR = Path("audio_files")          
MIME = {".wav": "audio/wav", ".mp3": "audio/mpeg"}
START_VOLUME = 0.5


def build_mixer(folder=SOUNDS_DIR, start_volume=START_VOLUME):
    mixer_id = "m" + uuid.uuid4().hex[:8]   
    rows = []

    for path in sorted(Path(folder).iterdir()):
        mime = MIME.get(path.suffix.lower())
        if mime is None:
            continue                        
        b64 = base64.b64encode(path.read_bytes()).decode()
        uid = "a" + uuid.uuid4().hex[:8]
        rows.append(f"""
        <div style="margin:10px 0">
          <b>{path.stem}</b><br>
          <audio id="{uid}" controls loop
                 onloadedmetadata="this.volume={start_volume}"
                 src="data:{mime};base64,{b64}"></audio><br>
          Volume: <input type="range" min="0" max="1" step="0.01" value="{start_volume}"
            oninput="document.getElementById('{uid}').volume=this.value"
            style="width:250px">
        </div>""")

    if not rows:
        print(f"No .wav or .mp3 files found in {Path(folder).resolve()}")
        return

    display(HTML(f"""
    <div id="{mixer_id}" style="font-family:sans-serif">
      <button onclick="const c=document.querySelectorAll('#{mixer_id} audio');
                       const p=c[0].paused;
                       c.forEach(a => p ? a.play() : a.pause());
                       this.textContent = p ? 'Pause all' : 'Play all'">Play all</button>
      {''.join(rows)}
    </div>"""))


build_mixer()